In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
#读取文件
housing=pd.read_csv('housing.csv')
#查看整体信息
housing.info()
#绘制直方图查看特征
housing.hist(bins=50,figsize=(12,8))
plt.show()

In [ ]:
#将收入中位数分成5个区间
housing['income_cat']=pd.cut(housing['median_income'],
                             bins=[0.,1.5,3.0,4.5,6.,np.inf],
                             labels=[1,2,3,4,5])
#按区间绘制收入中位数柱状图
housing['income_cat'].value_counts().sort_index().plot.bar(rot=0,grid=True)
plt.xlabel('Income category')
plt.ylabel('Number of districts')
plt.show()
#StratifiedShuffleSpilt类，作用是随机划分打乱的训练集与测试集，返回索引范围
from sklearn.model_selection import StratifiedShuffleSplit
splitter=StratifiedShuffleSplit(n_splits=10,test_size=0.2,random_state=42)
strat_splits=[]
#将10份划分好的索引范围分别记录，分割器的split（）方法
for train_index, test_index in splitter.split(housing,housing['income_cat']):
    strat_train_set_n=housing.iloc[train_index]
    strat_test_set_n=housing.iloc[test_index]
    strat_splits.append([strat_train_set_n,strat_test_set_n])
#选择第一份划分（其实不必使用循环记录选择，有更好的选择方式）
strat_train_set_n,strat_test_set_n=strat_splits[0]
#将不用的‘income_cat’特征去除
for set_ in(strat_train_set_n,strat_test_set_n):
    set_.drop('income_cat',axis=1,inplace=True)
#查看选择好的划分过的测试集前5行信息
strat_train_set_n.head()

In [ ]:
#将测试集进行复制，避免操作员原测试集
housing=strat_train_set_n.copy()
#绘制经纬度分布散点图
housing.plot(kind='scatter',x='longitude',y='latitude',grid=True)
plt.show()
#查看数据集前5行信息
housing.head()

In [ ]:
#绘制经纬度分布散点图，设置透明度alpha
housing.plot(kind='scatter',x='longitude',y='latitude',grid=True,alpha=0.2)
plt.show()

In [ ]:
#绘制经纬度散点图，更加丰富的可视化
housing.plot(kind='scatter',x='longitude',y='latitude',grid=True,
             s=housing['population']/100,label='population',
             c='median_house_value',cmap='jet',colorbar=True,
             legend=True,sharex=False,figsize=(10,7))
plt.show()

In [ ]:
#去除非数值类特征
housing.drop('ocean_proximity',axis=1,inplace=True)
#相关系数矩阵
corr_matrix=housing.corr()
#查看与房价中位数的各个特征相关系数
corr_matrix['median_house_value'].sort_values(ascending=False)

In [ ]:
#pandas查看选取特征组合关系图的方法
from pandas.plotting import scatter_matrix
attributes=['median_house_value','median_income','total_rooms',
            'housing_median_age']
scatter_matrix(housing[attributes],figsize=(12,8))
plt.show()

In [ ]:
#绘制收入中位数与房价中位数关系散点图
housing.plot(kind='scatter',x='median_income',y='median_house_value',grid=True,alpha=0.1,)
plt.show()

In [ ]:
#创建新特征，探索与房价中位数更为相关的特征
housing['room_per_house']=housing['total_rooms']/housing['households']
housing['bedrooms_ratio']=housing['total_bedrooms']/housing['total_rooms']
housing['people_per_house']=housing['population']/housing['households']
corr_matrix=housing.corr()
corr_matrix['median_house_value'].sort_values(ascending=False)

In [ ]:
#housing现在为去除目标值的训练集
housing=strat_train_set_n.drop('median_house_value',axis=1)
#训练集中的目标值
housing_lables=strat_train_set_n['median_house_value'].copy()

In [ ]:
#SimpleImputer类，对缺失值的处理
from sklearn.impute import SimpleImputer
imputer=SimpleImputer(strategy='median')
#housing_num为只包括数值类特征的数据
housing_num=housing.select_dtypes(include=[np.number])
#对所有housing_num的所有特征进行拟合
imputer.fit(housing_num)
#statistics_属性
print(imputer.statistics_)
#对所有特征填补缺失值
X=imputer.transform(housing_num)

In [ ]:
#两种处理非数值类特征的编码方式
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
#非数值类
housing_cat=housing[['ocean_proximity']]
housing_cat.head(8)
ordinal_encoder=OrdinalEncoder()
ordinal_encoder.fit(housing_cat)
housing_cat_encoded=ordinal_encoder.fit_transform(housing_cat)
housing_cat_encoded[:8]
#categories属性
ordinal_encoder.categories_
#独热编码，handle_unknown参数处理未知特征值
cat_encoder=OneHotEncoder(handle_unknown='ignore')
housing_cat_1hot=cat_encoder.fit_transform(housing_cat)
housing_cat_1hot
#categories属性
cat_encoder.categories_

In [ ]:
#OneHotEncoder类对比pandas中的get_dummies方法，更强大
df_test=pd.DataFrame({'ocean_proximity':['INLAND','NEAR BAY']})
print(pd.get_dummies(df_test))
print(cat_encoder.transform(df_test).toarray())
df_test_unknown=pd.DataFrame({'ocean_proximity':['<2H OCEAN','ISLAND']})
print(pd.get_dummies(df_test_unknown))
print(cat_encoder.transform(df_test_unknown).toarray())

In [ ]:
#实例属性，输入特征数，输入特征名，拟合后输出特征名
print(cat_encoder.n_features_in_)
print(cat_encoder.feature_names_in_)
print(cat_encoder.get_feature_names_out())

In [ ]:
#两种进行特征缩放的类，标准化与归一化
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
min_max_scaler=MinMaxScaler(feature_range=(-1,1))
housing_num_min_max_sacled=min_max_scaler.fit_transform(housing_num)
std_scaler=StandardScaler()
housing_num_std_scaled=std_scaler.fit_transform(housing_num)

In [ ]:
#rbf，高斯径向基函数，观察多峰分布
from sklearn.metrics.pairwise import rbf_kernel
age_simil_35=rbf_kernel(housing[['housing_median_age']],[[35]],gamma=0.1)

In [ ]:
#线性回归模型
from sklearn.linear_model import LinearRegression
#缩放器
target_scaler=StandardScaler()
#处理目标值
scaled_labels=target_scaler.fit_transform(housing_lables.to_frame())
#通过收入中位数演示对处理过的房价中位数的预测
model=LinearRegression()
model.fit(housing[['median_income']],scaled_labels)
some_new_data=housing[['median_income']].iloc[:5]
scaled_predictions=model.predict(some_new_data)
#特征值逆变换
predictions=target_scaler.inverse_transform(scaled_predictions)

In [ ]:
#封装了之前讨论过的 fit-transform和 inverse-transform逻辑的TransformedTargetRegressor类
from sklearn.compose import TransformedTargetRegressor
model=TransformedTargetRegressor(LinearRegression(),
                                 transformer=StandardScaler())
model.fit(housing[['median_income']],scaled_labels)
predictions=model.predict(some_new_data)

In [ ]:
#将任意 Python 函数包装成一个标准的 Transformer（转换器）的适配器，自定义转换器，FunctionTransformer类
from sklearn.preprocessing import FunctionTransformer
log_transformer=FunctionTransformer(np.log,inverse_func=np.exp)
log_pop=log_transformer.transform(housing[['population']])
sf_coords=37.7749,-122.41
rbf_transformer=FunctionTransformer(rbf_kernel,
                                    kw_args=dict(Y=[sf_coords],gamma=0.1))
ratio_transformer=FunctionTransformer(lambda x:x[:,[0]]/x[:,[1]])
ratio_transformer.transform(np.array([[1,2],[3,4]]))

In [ ]:
#自定义类
from sklearn.base import BaseEstimator,TransformerMixin
from sklearn.utils.validation import check_array,check_is_fitted

class StandardScalerClone(BaseEstimator,TransformerMixin):
    def __init__(self,with_mean=True):
        self.with_mean=with_mean

    def fit(self,X,y=None):
        X=check_array(X)
        self.mean_=X.mean(axis=0)
        self.scale_=X.std(axis=0)
        self.n_features_in_=X.shape[1]
        return self

    def transform(self,X):
        check_is_fitted(self)
        X=check_array(X)
        assert self.n_features_in_==X.shape[1]
        if self.with_mean:
            X=X-self.mean_
            return X/self.scale_
#未完成的自定义类

In [ ]:
#在自定义类中使用K均值发现各地区与集群中心的相似性
from sklearn.cluster import KMeans

class ClusterSimilarity(BaseEstimator,TransformerMixin):
    def __init__(self,n_clusters=10,gamma=1.0,random_state=42):
        self.n_clusters=n_clusters
        self.gamma=gamma
        self.random_state=random_state

    def fit(self,X,y=None,sample_weight=None):
        self.kmeans_=KMeans(self.n_clusters,random_state=self.random_state)
        self.kmeans_.fit(X,sample_weight=sample_weight)
        return self

    def transform(self,X):
        return rbf_kernel(X,self.kmeans_.cluster_centers_,self.gamma)

    def get_feature_names_out(self,names=None):
        return [f'Cluster {i} similarity' for i in range(self.n_clusters)]
#可通过将实例传递给'sklearn.utils.estimator_checks'包中的check_estimator()
# 来检查自定义估算器是否遵循sklearn的API

cluster_simil=ClusterSimilarity(n_clusters=10,gamma=1.0,random_state=42)
similarities=cluster_simil.fit_transform(housing[['latitude','longitude']],
                                        sample_weight=housing_lables
                                         )
similarities[:3].round(2)

In [ ]:
#pipeline流水线的构建
#make_pipeline自动命名
from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
num_pipeline=Pipeline([
    ('impute',SimpleImputer(strategy='median')),
    ('standardize',StandardScaler()),
])
num_pipeline=make_pipeline(SimpleImputer(strategy='median'),StandardScaler())
housing_num_prepared=num_pipeline.fit_transform(housing_num)
housing_num_prepared[:2].round(2)
df_housing_num_prepared=pd.DataFrame(
    housing_num_prepared,columns=num_pipeline.get_feature_names_out(),
    index=housing_num.index
)
print(df_housing_num_prepared)

In [ ]:
#允许对 DataFrame 的不同列应用不同的 Transformer的流水线‘工作站’，ColumnTransformer类
from sklearn.compose import ColumnTransformer
num_attribs=['longtitude','latitude','housing_median_age','total_rooms',
             'total_bedrooms','population','households','median_income']
cat_attribs=['ocean_proximity']
cat_pipeline=make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore')
)
#两条流水线拼接，数值类与非数值类分别执行不同处理
preprocessing=ColumnTransformer([
    ('num',num_pipeline,num_attribs),
    ('cat',cat_pipeline,cat_attribs),
])

In [ ]:
#作用与ColumnTransformer类类似，更加自动，操作更简单
from sklearn.compose import make_column_selector,make_column_transformer
preprocessing=make_column_transformer(
    (num_pipeline,make_column_selector(dtype_include=np.number)),
    (cat_pipeline,make_column_selector(dtype_include=object)),
)
housing_prepared=preprocessing.fit_transform(housing)

In [ ]:
#数据预处理流水线过程
def column_ratio(X):
    return X[:,[0]]/X[:,[1]]

def ratio_name(function_transformer,feature_names_in):
        return ['ratio']

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy='median'),
        FunctionTransformer(column_ratio,feature_names_out=ratio_name),
        StandardScaler())

log_pipeline=make_pipeline(
    SimpleImputer(strategy='median'),
    FunctionTransformer(np.log,feature_names_out='one-to-one'),
    StandardScaler())
cluster_simil=ClusterSimilarity(n_clusters=10,gamma=1.0,random_state=42)
default_num_pipeline=make_pipeline(SimpleImputer(strategy='median'),
                                   StandardScaler())

preprocessing=ColumnTransformer([
    ('bedrooms',ratio_pipeline(),['total_bedrooms','total_rooms']),
    ('rooms_per_house',ratio_pipeline(),['total_rooms','households']),
    ('people_per_houose',ratio_pipeline(),['population','households']),
    ('log',log_pipeline,['total_bedrooms','total_rooms','population',
                         'households','median_income']),
    ('geo',cluster_simil,['latitude','longitude']),
    ('cat',cat_pipeline,make_column_selector(dtype_include=object)),
],
remainder=default_num_pipeline)

housing_prepared=preprocessing.fit_transform(housing)
housing_prepared.shape
preprocessing.get_feature_names_out()

In [ ]:
#线性回归的数据预处理与模型调用流水线
lin_reg=make_pipeline(preprocessing,LinearRegression())
lin_reg.fit(housing,housing_lables)
housing_predictions=lin_reg.predict(housing)
print(housing_predictions[:5].round(-2))
print(housing_lables[:5].values)

In [ ]:
#使用误差分数评估模型
from sklearn.metrics import root_mean_squared_error
lin_rmse=root_mean_squared_error(housing_lables,housing_predictions)
print(lin_rmse)

In [ ]:
#决策树与交叉验证得分评估
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
tree_reg=make_pipeline(preprocessing,DecisionTreeRegressor(random_state=42))
tree_reg.fit(housing,housing_lables)
housing_predictions=tree_reg.predict(housing)
tree_rmse=root_mean_squared_error(housing_lables,housing_predictions)
print(tree_rmse)
tree_rmses=-cross_val_score(tree_reg,housing,housing_lables,
                            scoring='neg_root_mean_squared_error',cv=10)
pd.Series(tree_rmses).describe()

In [ ]:
#随机森林优秀性
from sklearn.ensemble import RandomForestRegressor
forest_reg=make_pipeline(preprocessing,RandomForestRegressor(random_state=42))
forest_rmses=-cross_val_score(forest_reg,housing,housing_lables,
                              scoring='neg_root_mean_squared_error',cv=10)


In [ ]:
pd.Series(forest_rmses).describe()

In [ ]:
#微调模型参数，网格搜索
from sklearn.model_selection import GridSearchCV
full_pipeline=Pipeline([
    ('preprocessing',preprocessing),
    ('random_forest',RandomForestRegressor(random_state=42))
])
param_grid=[
    {'preprocessing__geo__n_clusters':[5,8,10],
     'random_forest__max_features':[4,6,8]},
    {'preprocessing__geo__n_clusters':[10,15],
     'random_forest__max_features':[6,8,10]},
]
grid_search=GridSearchCV(full_pipeline,param_grid,cv=3,
                         scoring='neg_root_mean_squared_error')
grid_search.fit(housing,housing_lables)
print(grid_search.best_params_)
cv_res=pd.DataFrame(grid_search.cv_results_)
cv_res.sort_values(by='mean_test_score',ascending=False,inplace=True)
cv_res.head()

In [ ]:
#微调模型参数，随机搜索
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
param_distribs={'preprocessing__geo__n_clusters':randint(low=3,high=50),
                'random_forest__max_features':randint(low=2,high=20)}
rnd_search=RandomizedSearchCV(
    full_pipeline,param_distributions=param_distribs,n_iter=10,cv=3,
    scoring='neg_root_mean_squared_error',random_state=42)

rnd_search.fit(housing,housing_lables)

In [ ]:
#最终模型，best_estimator_实例属性
final_model=rnd_search.best_estimator_
#随机森林的特征权重分析，可排除无用特征
feature_importances=final_model['random_forest'].feature_importances_
feature_importances.round(2)
sorted(zip(feature_importances,
       final_model['preprocessing'].get_feature_names_out()),
       reverse=True)

In [ ]:
#对房价进行预测，对模型进行评估
X_test=strat_test_set_n.drop('median_house_value',axis=1)
y_test=strat_test_set_n['median_house_value'].copy()
final_predictions=final_model.predict(X_test)
final_rmse=root_mean_squared_error(y_test,final_predictions)
print(final_rmse)